In [1]:
def main(datasources, start_date, end_date):
    """AI051: event-time impact hysteresis factor with regression/ranking ensemble."""
    import gc
    import sys
    import time

    main_t0 = time.time()

    def _rss_mb():
        try:
            import resource

            rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            if sys.platform == "darwin":
                return round(rss / 1024.0 / 1024.0, 2)
            return round(rss / 1024.0, 2)
        except Exception:
            return None

    def _df_mb(df):
        try:
            return round(float(df.memory_usage(deep=True).sum()) / 1048576.0, 2)
        except Exception:
            return None

    def _log(event, **kwargs):
        parts = [f"[AI051] {event}", f"elapsed={time.time() - main_t0:.2f}s"]
        rss = _rss_mb()
        if rss is not None:
            parts.append(f"rss_peak_mb={rss}")
        parts.extend(f"{key}={value}" for key, value in kwargs.items())
        print(" | ".join(parts), flush=True)

    _log("main_enter", start_date=start_date, end_date=end_date)
    import numpy as np
    import pandas as pd
    import dai

    _log("xgboost_import_start")
    import xgboost as xgb

    _log("imports_done", xgboost_version=getattr(xgb, "__version__", "unknown"))

    TRAIN_START = "2021-01-01 00:00:00"
    OFFICIAL_TRAIN_END = "2024-12-31 23:59:59"
    TRAIN_BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
    HISTORY_CALENDAR_DAYS = 35
    RANDOM_SEED = 20260851
    MAX_TRAIN_DATES = 760
    EVENT_BUCKETS = 10
    RAW_METRICS = (
        "duration", "ret", "spread", "depth", "obi", "order_imbalance",
        "trade_size", "signed_pressure", "impact_efficiency", "close_location",
    )
    PATH_FEATURES = [
        "clock_entropy", "clock_hhi", "clock_front", "clock_tail",
        "clock_acceleration", "clock_asymmetry", "clock_market_deviation",
        "return_path_efficiency", "return_path_reversal", "return_convexity",
        "permanent_impact_ratio", "temporary_impact_peak", "impact_hysteresis",
        "signed_pressure_total", "pressure_price_alignment", "pressure_persistence",
        "pressure_reversal", "spread_shock", "spread_recovery", "depth_shock",
        "depth_replenishment", "obi_price_alignment", "order_price_alignment",
        "liquidity_response_gap", "informed_impact", "uninformed_churn",
        "market_clock_residual", "market_impact_residual", "market_pressure_residual",
        "day_return", "range_amplitude", "close_position", "total_amount_log",
        "total_deals_log", "minute_burstiness", "minute_amount_hhi",
    ]
    PATH_FEATURES += [
        f"e{idx:02d}_{metric}"
        for idx in range(1, EVENT_BUCKETS + 1)
        for metric in RAW_METRICS
    ]
    HISTORY_SOURCES = [
        "clock_acceleration", "impact_hysteresis", "permanent_impact_ratio",
        "signed_pressure_total", "pressure_price_alignment", "depth_replenishment",
        "informed_impact", "market_impact_residual",
    ]

    def chunk_ranges(start, end):
        current = pd.Timestamp(start).normalize()
        finish = pd.Timestamp(end).normalize()
        while current <= finish:
            chunk_end = min(current + pd.Timedelta(days=15), finish)
            yield current, chunk_end
            current = chunk_end + pd.Timedelta(days=1)

    def rank_cross_section(df, columns, phase):
        _log("rank_features_start", phase=phase, rows=len(df), features=len(columns))
        for col in columns:
            values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            values = values.fillna(df.groupby("date", sort=False)[col].transform("median"))
            df[col] = values.groupby(df["date"], sort=False).rank(pct=True).sub(0.5)
            df[col] = df[col].fillna(0.0).astype("float32")
        _log("rank_features_done", phase=phase, df_mb=_df_mb(df))
        return df

    def model_parameters(n_estimators, objective):
        return dict(
            n_estimators=int(n_estimators), max_depth=5, learning_rate=0.035,
            subsample=0.82, colsample_bytree=0.76, min_child_weight=120,
            reg_alpha=1.0, reg_lambda=28.0, gamma=0.015,
            objective=objective, tree_method="hist", max_bin=192, n_jobs=4,
            random_state=RANDOM_SEED,
        )

    def event_aggregate_sql():
        expressions = []
        for idx in range(1, EVENT_BUCKETS + 1):
            name = f"e{idx:02d}"
            expressions.extend([
                f"COUNT(*) FILTER (WHERE event_bucket = {idx}) AS {name}_bars",
                f"MIN(minute_of_day) FILTER (WHERE event_bucket = {idx}) AS {name}_first_minute",
                f"MAX(minute_of_day) FILTER (WHERE event_bucket = {idx}) AS {name}_last_minute",
                f"ARG_MIN(mid_price, date) FILTER (WHERE event_bucket = {idx}) AS {name}_first_mid",
                f"ARG_MAX(mid_price, date) FILTER (WHERE event_bucket = {idx}) AS {name}_last_mid",
                f"MIN(mid_price) FILTER (WHERE event_bucket = {idx}) AS {name}_low_mid",
                f"MAX(mid_price) FILTER (WHERE event_bucket = {idx}) AS {name}_high_mid",
                f"SUM(delta_amount) FILTER (WHERE event_bucket = {idx}) AS {name}_amount",
                f"SUM(delta_deals) FILTER (WHERE event_bucket = {idx}) AS {name}_deals",
                f"AVG(rel_spread) FILTER (WHERE event_bucket = {idx}) AS {name}_spread",
                f"AVG(total_depth) FILTER (WHERE event_bucket = {idx}) AS {name}_depth",
                f"SUM(obi * delta_amount) FILTER (WHERE event_bucket = {idx}) "
                f"/ (SUM(delta_amount) FILTER (WHERE event_bucket = {idx}) + 1e-8) AS {name}_obi",
                f"SUM(order_imbalance * delta_amount) FILTER (WHERE event_bucket = {idx}) "
                f"/ (SUM(delta_amount) FILTER (WHERE event_bucket = {idx}) + 1e-8) "
                f"AS {name}_order_imbalance",
                f"SUM(signed_pressure) FILTER (WHERE event_bucket = {idx}) AS {name}_pressure",
                f"SUM(abs(minute_ret)) FILTER (WHERE event_bucket = {idx}) AS {name}_path_length",
            ])
        return ",\n                    ".join(expressions)

    def event_output_sql():
        expressions = []
        for idx in range(1, EVENT_BUCKETS + 1):
            name = f"e{idx:02d}"
            expressions.extend([
                f"({name}_last_minute - {name}_first_minute + 1.0) / 240.0 AS {name}_duration",
                f"{name}_last_mid / ({name}_first_mid + 1e-8) - 1.0 AS {name}_ret",
                f"{name}_spread AS {name}_spread",
                f"log(1.0 + {name}_depth) AS {name}_depth",
                f"{name}_obi AS {name}_obi",
                f"{name}_order_imbalance AS {name}_order_imbalance",
                f"log(1.0 + {name}_amount / ({name}_deals + 1.0)) AS {name}_trade_size",
                f"{name}_pressure / (sqrt({name}_amount) + 1.0) AS {name}_signed_pressure",
                f"abs(log({name}_last_mid / ({name}_first_mid + 1e-8))) "
                f"/ ({name}_path_length + 1e-8) AS {name}_impact_efficiency",
                f"({name}_last_mid - {name}_low_mid) "
                f"/ ({name}_high_mid - {name}_low_mid + 1e-8) AS {name}_close_location",
            ])
        return ",\n                ".join(expressions)

    def build_event_sql(table, q_start, q_end):
        event_aggs = event_aggregate_sql()
        event_outputs = event_output_sql()
        bucket_checks = " AND ".join(
            f"e{idx:02d}_bars >= 1" for idx in range(1, EVENT_BUCKETS + 1)
        )
        return f"""
        WITH base AS (
            SELECT
                date, instrument, date_trunc('day', date)::DATE AS trading_day,
                CAST(open AS DOUBLE) AS open, CAST(high AS DOUBLE) AS high,
                CAST(low AS DOUBLE) AS low, CAST(close AS DOUBLE) AS close,
                CAST(pre_close AS DOUBLE) AS pre_close,
                CAST(amount AS DOUBLE) AS amount, CAST(deal_number AS DOUBLE) AS deal_number,
                (CAST(ask_price1 AS DOUBLE) + CAST(bid_price1 AS DOUBLE)) / 2.0 AS mid_price,
                (CAST(ask_price1 AS DOUBLE) - CAST(bid_price1 AS DOUBLE))
                    / ((CAST(ask_price1 AS DOUBLE) + CAST(bid_price1 AS DOUBLE)) / 2.0 + 1e-8)
                    AS rel_spread,
                CAST(COALESCE(bid_volume1, 0) AS DOUBLE) + CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_volume3, 0) AS DOUBLE) + CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_volume5, 0) AS DOUBLE) AS bid_depth,
                CAST(COALESCE(ask_volume1, 0) AS DOUBLE) + CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_volume3, 0) AS DOUBLE) + CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_volume5, 0) AS DOUBLE) AS ask_depth,
                CAST(COALESCE(bid_num_orders1, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_num_orders2, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_num_orders3, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_num_orders4, 0) AS DOUBLE)
                    + CAST(COALESCE(bid_num_orders5, 0) AS DOUBLE) AS bid_orders,
                CAST(COALESCE(ask_num_orders1, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_num_orders2, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_num_orders3, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_num_orders4, 0) AS DOUBLE)
                    + CAST(COALESCE(ask_num_orders5, 0) AS DOUBLE) AS ask_orders
            FROM {table}
            WHERE date >= CAST('{q_start}' AS DATETIME) AND date <= CAST('{q_end}' AS DATETIME)
              AND strftime(date, '%H:%M:%S') >= '09:31:00'
              AND strftime(date, '%H:%M:%S') <= '15:00:00'
              AND close > 0 AND ask_price1 > 0 AND bid_price1 > 0 AND ask_price1 >= bid_price1
        ),
        lagged AS (
            SELECT *,
                lag(mid_price) OVER (PARTITION BY instrument, trading_day ORDER BY date) AS prev_mid,
                lag(amount) OVER (PARTITION BY instrument, trading_day ORDER BY date) AS prev_amount,
                lag(deal_number) OVER (PARTITION BY instrument, trading_day ORDER BY date) AS prev_deals,
                CASE
                    WHEN strftime(date, '%H:%M:%S') <= '11:30:00'
                    THEN date_diff('minute', date_trunc('day', date) + INTERVAL 9 HOUR
                        + INTERVAL 30 MINUTE, date)
                    ELSE 120 + date_diff('minute', date_trunc('day', date) + INTERVAL 13 HOUR, date)
                END AS minute_of_day
            FROM base
        ),
        cleaned AS (
            SELECT *,
                CASE
                    WHEN amount IS NULL THEN 0.0
                    WHEN prev_amount IS NULL OR amount < prev_amount THEN greatest(amount, 0.0)
                    ELSE greatest(amount - prev_amount, 0.0)
                END AS delta_amount,
                CASE
                    WHEN deal_number IS NULL THEN 0.0
                    WHEN prev_deals IS NULL OR deal_number < prev_deals THEN greatest(deal_number, 0.0)
                    ELSE greatest(deal_number - prev_deals, 0.0)
                END AS delta_deals,
                CASE WHEN prev_mid > 0 THEN log(mid_price / prev_mid) ELSE 0.0 END AS minute_ret,
                bid_depth + ask_depth AS total_depth,
                (bid_depth - ask_depth) / (bid_depth + ask_depth + 1e-8) AS obi,
                (bid_orders - ask_orders) / (bid_orders + ask_orders + 1e-8) AS order_imbalance
            FROM lagged
        ),
        cumulative AS (
            SELECT *,
                SUM(delta_amount) OVER (
                    PARTITION BY instrument, trading_day ORDER BY date
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS cumulative_amount,
                SUM(delta_amount) OVER (PARTITION BY instrument, trading_day) AS total_amount,
                SUM(delta_deals) OVER (PARTITION BY instrument, trading_day) AS total_deals
            FROM cleaned
        ),
        bucketed AS (
            SELECT *,
                LEAST(10, GREATEST(1, CAST(FLOOR(
                    10.0 * greatest(cumulative_amount - 0.5 * delta_amount, 0.0)
                    / (total_amount + 1e-8)
                ) AS INTEGER) + 1)) AS event_bucket,
                sign(minute_ret) * sqrt(delta_amount) AS signed_pressure
            FROM cumulative
            WHERE total_amount > 0 AND total_deals > 0
        ),
        daily AS (
            SELECT
                trading_day, instrument, COUNT(*) AS day_bars,
                ARG_MIN(open, date) AS first_open, ARG_MIN(pre_close, date) AS day_pre_close,
                ARG_MAX(mid_price, date) AS daily_close,
                MAX(high) AS high_price, MIN(low) AS low_price,
                MAX(total_amount) AS day_amount, MAX(total_deals) AS day_deals,
                STDDEV_POP(delta_amount) / (AVG(delta_amount) + 1e-8) AS minute_burstiness,
                SUM(delta_amount * delta_amount) / power(MAX(total_amount) + 1e-8, 2)
                    AS minute_amount_hhi,
                {event_aggs}
            FROM bucketed
            GROUP BY trading_day, instrument
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date, instrument, daily_close,
            daily_close / (day_pre_close + 1e-8) - 1.0 AS day_return,
            high_price / (low_price + 1e-8) - 1.0 AS range_amplitude,
            (daily_close - low_price) / (high_price - low_price + 1e-8) AS close_position,
            log(1.0 + day_amount) AS total_amount_log,
            log(1.0 + day_deals) AS total_deals_log,
            minute_burstiness, minute_amount_hhi,
            {event_outputs}
        FROM daily
        WHERE day_bars >= 120 AND day_amount > 0 AND day_deals > 0
          AND day_pre_close > 0 AND first_open > 0 AND high_price > low_price
          AND {bucket_checks}
        """

    def query_event_features(table, sd, ed, phase):
        query_t0 = time.time()
        _log("pool_preload_start", phase=phase, start=sd, end=ed)
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]}, compression=True,
        ).df()
        pool["date"] = pd.to_datetime(pool["date"]).dt.normalize()
        pool["instrument"] = pool["instrument"].astype(str)
        pool = pool.drop_duplicates(["date", "instrument"])
        _log("pool_preload_done", phase=phase, rows=len(pool), df_mb=_df_mb(pool))
        frames = []
        for chunk_no, (chunk_start, chunk_end) in enumerate(chunk_ranges(sd, ed), 1):
            q_start = f"{chunk_start.date()} 00:00:00"
            q_end = f"{chunk_end.date()} 23:59:59"
            chunk_t0 = time.time()
            _log("feature_chunk_start", phase=phase, chunk=chunk_no, start=q_start, end=q_end)
            chunk = dai.query(
                build_event_sql(table, q_start, q_end),
                filters={"date": [q_start, q_end]}, compression=True,
            ).df()
            raw_rows = len(chunk)
            if not chunk.empty:
                chunk["date"] = pd.to_datetime(chunk["date"]).dt.normalize()
                chunk["instrument"] = chunk["instrument"].astype(str)
                chunk_pool = pool[(pool["date"] >= chunk_start) & (pool["date"] <= chunk_end)]
                chunk = pd.merge(chunk_pool, chunk, how="inner", on=["date", "instrument"])
                numeric_cols = chunk.columns.difference(["date", "instrument"])
                chunk[numeric_cols] = chunk[numeric_cols].astype("float32")
                frames.append(chunk)
            _log(
                "feature_chunk_done", phase=phase, chunk=chunk_no, raw_rows=raw_rows,
                rows=len(chunk), seconds=round(time.time() - chunk_t0, 2), df_mb=_df_mb(chunk),
            )
        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        del frames, pool
        gc.collect()
        if df.empty:
            raise ValueError(f"{phase} 事件时间特征为空，请检查日期和bar1m数据源")
        df = df.drop_duplicates(["date", "instrument"], keep="last")
        df = df.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)
        _log(
            "feature_chunks_done", phase=phase, rows=len(df), cols=len(df.columns),
            date_min=df["date"].min(), date_max=df["date"].max(),
            seconds=round(time.time() - query_t0, 2), df_mb=_df_mb(df),
        )
        return df

    def add_path_features(df, phase):
        feature_t0 = time.time()
        _log("path_features_start", phase=phase, rows=len(df), df_mb=_df_mb(df))
        raw_cols = [
            f"e{idx:02d}_{metric}"
            for idx in range(1, EVENT_BUCKETS + 1)
            for metric in RAW_METRICS
        ]
        daily_cols = [
            "day_return", "range_amplitude", "close_position", "total_amount_log",
            "total_deals_log", "minute_burstiness", "minute_amount_hhi",
        ]
        for col in raw_cols + daily_cols:
            values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[col] = values.fillna(df.groupby("date", sort=False)[col].transform("median"))
            df[col] = df[col].fillna(0.0).astype("float32")

        def matrix(metric):
            return df[
                [f"e{idx:02d}_{metric}" for idx in range(1, EVENT_BUCKETS + 1)]
            ].to_numpy(dtype=np.float32)

        def row_corr(left, right):
            left = left - left.mean(axis=1, keepdims=True)
            right = right - right.mean(axis=1, keepdims=True)
            scale = np.sqrt(np.sum(left * left, axis=1) * np.sum(right * right, axis=1)) + 1e-8
            return np.sum(left * right, axis=1) / scale

        duration_raw = np.maximum(matrix("duration"), 1.0 / 240.0)
        clock = duration_raw / (duration_raw.sum(axis=1, keepdims=True) + 1e-8)
        ret = matrix("ret")
        spread = np.maximum(matrix("spread"), 0.0)
        depth = matrix("depth")
        obi = matrix("obi")
        order_imbalance = matrix("order_imbalance")
        pressure = matrix("signed_pressure")
        efficiency = np.clip(matrix("impact_efficiency"), 0.0, 1.5)
        cumulative_return = np.cumsum(ret, axis=1)
        cumulative_clock = np.cumsum(clock, axis=1)
        final_return = cumulative_return[:, -1]
        max_abs_return = np.max(np.abs(cumulative_return), axis=1) + 1e-8
        peak_index = np.argmax(np.abs(cumulative_return), axis=1)
        peak_return = cumulative_return[np.arange(len(df)), peak_index]
        market_clock = np.column_stack([
            df.groupby("date", sort=False)[f"e{idx:02d}_duration"].transform("median")
            .to_numpy(dtype=np.float32)
            for idx in range(1, EVENT_BUCKETS + 1)
        ])
        market_clock = market_clock / (market_clock.sum(axis=1, keepdims=True) + 1e-8)
        market_ret = np.column_stack([
            df.groupby("date", sort=False)[f"e{idx:02d}_ret"].transform("median")
            .to_numpy(dtype=np.float32)
            for idx in range(1, EVENT_BUCKETS + 1)
        ])
        market_pressure = np.column_stack([
            df.groupby("date", sort=False)[f"e{idx:02d}_signed_pressure"].transform("median")
            .to_numpy(dtype=np.float32)
            for idx in range(1, EVENT_BUCKETS + 1)
        ])
        market_final_return = market_ret.sum(axis=1)
        duration_index = np.linspace(0.0, 1.0, EVENT_BUCKETS, dtype=np.float32)
        clock_residual = clock - market_clock
        path_length = np.sum(np.abs(ret), axis=1) + 1e-8
        pressure_abs = np.sum(np.abs(pressure), axis=1) + 1e-8
        spread_change = np.diff(spread, axis=1)
        depth_change = np.diff(depth, axis=1)
        liquidity_response = row_corr(pressure[:, :-1], depth_change - spread_change)
        values = {
            "clock_entropy": -np.sum(clock * np.log(clock + 1e-8), axis=1),
            "clock_hhi": np.sum(clock * clock, axis=1),
            "clock_front": clock[:, :3].sum(axis=1) - 0.30,
            "clock_tail": clock[:, -2:].sum(axis=1) - 0.20,
            "clock_acceleration": np.log(
                (clock[:, :3].mean(axis=1) + 1e-6)
                / (clock[:, -3:].mean(axis=1) + 1e-6)
            ),
            "clock_asymmetry": np.sum(clock * (duration_index - 0.5), axis=1),
            "clock_market_deviation": np.sqrt(np.sum(clock_residual ** 2, axis=1)),
            "return_path_efficiency": np.abs(final_return) / path_length,
            "return_path_reversal": final_return - peak_return,
            "return_convexity": ret[:, -5:].sum(axis=1) - ret[:, :5].sum(axis=1),
            "permanent_impact_ratio": np.abs(final_return) / max_abs_return,
            "temporary_impact_peak": max_abs_return - np.abs(final_return),
            "impact_hysteresis": np.sum(
                0.5 * (cumulative_return[:, 1:] + cumulative_return[:, :-1])
                * np.diff(cumulative_clock, axis=1), axis=1,
            ),
            "signed_pressure_total": pressure.sum(axis=1),
            "pressure_price_alignment": row_corr(pressure, ret),
            "pressure_persistence": row_corr(pressure[:, 1:], pressure[:, :-1]),
            "pressure_reversal": pressure[:, -3:].mean(axis=1) - pressure[:, :3].mean(axis=1),
            "spread_shock": spread.max(axis=1) - spread[:, 0],
            "spread_recovery": spread.max(axis=1) - spread[:, -2:].mean(axis=1),
            "depth_shock": depth.min(axis=1) - depth[:, 0],
            "depth_replenishment": depth[:, -2:].mean(axis=1) - depth.min(axis=1),
            "obi_price_alignment": row_corr(obi, ret),
            "order_price_alignment": row_corr(order_imbalance, ret),
            "liquidity_response_gap": liquidity_response,
            "informed_impact": row_corr(pressure, ret)
                * efficiency.mean(axis=1) * np.abs(final_return) / path_length,
            "uninformed_churn": pressure_abs * (1.0 - np.abs(final_return) / path_length),
            "market_clock_residual": np.sum(clock_residual * duration_index, axis=1),
            "market_impact_residual": final_return - market_final_return,
            "market_pressure_residual": pressure.sum(axis=1) - market_pressure.sum(axis=1),
        }
        feature_frame = pd.DataFrame(index=df.index)
        for col, value in values.items():
            feature_frame[col] = np.nan_to_num(
                value, nan=0.0, posinf=0.0, neginf=0.0
            ).astype("float32")
        df = pd.concat([df, feature_frame], axis=1).copy()
        _log(
            "path_features_done", phase=phase, rows=len(df), features=len(PATH_FEATURES),
            seconds=round(time.time() - feature_t0, 2), df_mb=_df_mb(df),
        )
        return df

    def add_history_features(df, phase):
        history_t0 = time.time()
        df = df.sort_values(["instrument", "date"], kind="mergesort").reset_index(drop=True)
        history_data = {}
        history_cols = []
        instrument_key = df["instrument"]
        for col in HISTORY_SOURCES:
            shifted = df.groupby("instrument", sort=False)[col].shift(1)
            lag_col = f"{col}_lag1"
            mean_col = f"{col}_mean5"
            surprise_col = f"{col}_surprise5"
            rolling_mean = shifted.groupby(instrument_key, sort=False).transform(
                lambda x: x.rolling(5, min_periods=2).mean()
            )
            history_data[lag_col] = shifted.astype("float32")
            history_data[mean_col] = rolling_mean.astype("float32")
            history_data[surprise_col] = (df[col] - rolling_mean).astype("float32")
            history_cols.extend([lag_col, mean_col, surprise_col])
        df = pd.concat([df, pd.DataFrame(history_data, index=df.index)], axis=1).copy()
        _log(
            "history_features_done", phase=phase, rows=len(df), features=len(history_cols),
            seconds=round(time.time() - history_t0, 2), df_mb=_df_mb(df),
        )
        return df, history_cols

    def prepare_dataset(table, query_start, query_end, crop_start, crop_end, phase):
        frame = query_event_features(table, query_start, query_end, phase)
        frame = add_path_features(frame, phase)
        frame, history_cols = add_history_features(frame, phase)
        frame = frame[
            (frame["date"] >= pd.Timestamp(crop_start).normalize())
            & (frame["date"] <= pd.Timestamp(crop_end).normalize())
        ].copy()
        model_cols = PATH_FEATURES + history_cols
        frame = rank_cross_section(frame, model_cols, phase)
        _log(
            "dataset_ready", phase=phase, rows=len(frame), features=len(model_cols),
            date_min=frame["date"].min(), date_max=frame["date"].max(), df_mb=_df_mb(frame),
        )
        return frame, model_cols

    def keep_recent_dates(df, max_dates):
        dates = np.array(sorted(df["date"].unique()))
        if len(dates) <= max_dates:
            return df
        cutoff = pd.Timestamp(dates[-max_dates])
        return df[df["date"] >= cutoff].copy()

    def group_sizes(df):
        return df.groupby("date", sort=True).size().to_numpy(dtype=np.int32)

    def cross_sectional_rank(dates, prediction):
        frame = pd.DataFrame({"date": dates, "prediction": prediction})
        return frame.groupby("date", sort=False)["prediction"].rank(pct=True).sub(0.5).to_numpy()

    def validation_metrics(frame, prediction):
        evaluated = frame[["date", "label", "raw_label"]].copy()
        evaluated["prediction"] = cross_sectional_rank(frame["date"], prediction)
        daily_ic = evaluated.groupby("date", sort=True)[["label", "prediction"]].apply(
            lambda x: x["label"].corr(x["prediction"])
        ).dropna()
        if daily_ic.empty:
            return {
                "ic": 0.0, "ir": 0.0, "positive": 0.0, "tail": 0.0,
                "stress": 0.0, "worst_block": 0.0,
            }
        daily_market = evaluated.groupby("date", sort=True)["raw_label"].mean()
        stress_dates = daily_market.nsmallest(max(20, len(daily_market) // 5)).index
        stress_ic = daily_ic.reindex(stress_dates).dropna()

        def tail_spread(day):
            ranked = day["prediction"].rank(pct=True)
            top = day.loc[ranked >= 0.9, "label"].mean()
            bottom = day.loc[ranked <= 0.1, "label"].mean()
            return top - bottom

        tails = evaluated.groupby("date", sort=True).apply(tail_spread).dropna()
        block_id = np.arange(len(daily_ic)) // 40
        block_ic = daily_ic.groupby(block_id).mean()
        return {
            "ic": float(daily_ic.mean()),
            "ir": float(daily_ic.mean() / (daily_ic.std(ddof=1) + 1e-8)),
            "positive": float((daily_ic > 0).mean()),
            "tail": float(tails.mean()) if not tails.empty else 0.0,
            "stress": float(stress_ic.mean()) if not stress_ic.empty else 0.0,
            "worst_block": float(block_ic.min()) if not block_ic.empty else 0.0,
        }

    def selection_score(metrics):
        return (
            metrics["ic"] + 0.03 * metrics["ir"]
            + 0.02 * (metrics["positive"] - 0.5)
            + 0.15 * metrics["tail"] + 0.35 * metrics["stress"]
            + 0.25 * metrics["worst_block"]
        )

    test_start = pd.Timestamp(start_date)
    test_end = pd.Timestamp(end_date)
    train_start = pd.Timestamp(TRAIN_START)
    train_end = min(pd.Timestamp(OFFICIAL_TRAIN_END), test_start - pd.Timedelta(days=1))
    if train_end < train_start:
        raise ValueError("训练区间为空：测试开始日期必须晚于2021-01-01")
    train_query_start = train_start - pd.Timedelta(days=HISTORY_CALENDAR_DAYS)
    test_query_start = test_start - pd.Timedelta(days=HISTORY_CALENDAR_DAYS)
    _log(
        "windows_resolved", train_start=train_start, train_end=train_end,
        train_query_start=train_query_start, test_start=test_start, test_end=test_end,
    )

    _log("train_dataset_start")
    train_df, model_feature_cols = prepare_dataset(
        TRAIN_BAR1M_TABLE,
        train_query_start.strftime("%Y-%m-%d 00:00:00"),
        train_end.strftime("%Y-%m-%d 23:59:59"),
        train_start, train_end, "train",
    )
    train_df = train_df.sort_values(["instrument", "date"], kind="mergesort")
    train_df["next_close"] = train_df.groupby("instrument", sort=False)["daily_close"].shift(-1)
    train_df["raw_label"] = train_df["next_close"] / train_df["daily_close"] - 1.0
    train_df["label"] = train_df.groupby("date", sort=False)["raw_label"].rank(pct=True).sub(0.5)
    train_df = train_df.dropna(subset=["label", "raw_label"]).copy()
    train_df["rank_label"] = np.floor((train_df["label"] + 0.5) * 5.0).clip(0, 4)
    train_df = keep_recent_dates(train_df, MAX_TRAIN_DATES)
    keep_cols = [
        "date", "instrument", "label", "rank_label", "raw_label"
    ] + model_feature_cols
    train_df = train_df[keep_cols].sort_values(
        ["date", "instrument"], kind="mergesort"
    ).reset_index(drop=True)
    _log(
        "train_labels_ready", rows=len(train_df), dates=train_df["date"].nunique(),
        features=len(model_feature_cols), label_mean=round(float(train_df["label"].mean()), 6),
        df_mb=_df_mb(train_df),
    )

    unique_dates = np.array(sorted(train_df["date"].unique()))
    validation_days = min(180, max(120, len(unique_dates) // 4))
    if len(unique_dates) <= validation_days + 40:
        raise ValueError("训练交易日不足，无法建立purged时间验证集")
    validation_start = pd.Timestamp(unique_dates[-validation_days])
    purge_date = pd.Timestamp(unique_dates[-validation_days - 1])
    fit_part = train_df[train_df["date"] < purge_date].copy()
    validation_part = train_df[train_df["date"] >= validation_start].copy()
    fit_part = fit_part.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)
    validation_part = validation_part.sort_values(
        ["date", "instrument"], kind="mergesort"
    ).reset_index(drop=True)
    fit_groups = group_sizes(fit_part)
    validation_groups = group_sizes(validation_part)
    _log(
        "validation_split_ready", fit_rows=len(fit_part), validation_rows=len(validation_part),
        fit_dates=len(fit_groups), validation_dates=len(validation_groups),
        validation_start=validation_start, purge_date=purge_date,
    )

    reg_validation = xgb.XGBRegressor(
        **model_parameters(900, "reg:squarederror"), eval_metric="rmse"
    )
    fit_weights = (1.0 + 1.0 * np.abs(fit_part["label"].to_numpy())).astype("float32")
    validation_weights = (
        1.0 + 1.0 * np.abs(validation_part["label"].to_numpy())
    ).astype("float32")
    fit_t0 = time.time()
    _log("reg_validation_fit_start", rows=len(fit_part), features=len(model_feature_cols))
    reg_validation.fit(
        fit_part[model_feature_cols], fit_part["label"], sample_weight=fit_weights,
        eval_set=[(validation_part[model_feature_cols], validation_part["label"])],
        sample_weight_eval_set=[validation_weights], early_stopping_rounds=60, verbose=50,
    )
    reg_best = getattr(reg_validation, "best_iteration", None)
    reg_trees = 900 if reg_best is None else int(reg_best) + 1
    reg_trees = int(np.clip(reg_trees, 140, 900))
    reg_prediction = reg_validation.predict(validation_part[model_feature_cols])
    _log(
        "reg_validation_fit_done", seconds=round(time.time() - fit_t0, 2),
        selected_trees=reg_trees,
    )

    rank_validation = xgb.XGBRanker(
        **model_parameters(700, "rank:pairwise"), eval_metric="ndcg@100"
    )
    fit_t0 = time.time()
    _log(
        "rank_validation_fit_start", rows=len(fit_part), groups=len(fit_groups),
        features=len(model_feature_cols),
    )
    rank_validation.fit(
        fit_part[model_feature_cols], fit_part["rank_label"], group=fit_groups,
        eval_set=[(validation_part[model_feature_cols], validation_part["rank_label"])],
        eval_group=[validation_groups], early_stopping_rounds=50, verbose=50,
    )
    rank_best = getattr(rank_validation, "best_iteration", None)
    rank_trees = 700 if rank_best is None else int(rank_best) + 1
    rank_trees = int(np.clip(rank_trees, 120, 700))
    rank_prediction = rank_validation.predict(validation_part[model_feature_cols])
    _log(
        "rank_validation_fit_done", seconds=round(time.time() - fit_t0, 2),
        selected_trees=rank_trees,
    )

    reg_rank = cross_sectional_rank(validation_part["date"], reg_prediction)
    pair_rank = cross_sectional_rank(validation_part["date"], rank_prediction)
    blend_results = []
    for reg_weight in (0.0, 0.25, 0.50, 0.75, 1.0):
        blended = reg_weight * reg_rank + (1.0 - reg_weight) * pair_rank
        metrics = validation_metrics(validation_part, blended)
        score = selection_score(metrics)
        blend_results.append((score, reg_weight, metrics))
        _log(
            "validation_blend", reg_weight=reg_weight, score=round(score, 6),
            ic=round(metrics["ic"], 6), ir=round(metrics["ir"], 6),
            positive=round(metrics["positive"], 6), tail=round(metrics["tail"], 6),
            stress=round(metrics["stress"], 6),
            worst_block=round(metrics["worst_block"], 6),
        )
    _, selected_reg_weight, selected_metrics = max(blend_results, key=lambda item: item[0])
    _log(
        "validation_selection_done", selected_reg_weight=selected_reg_weight,
        selected_rank_weight=1.0 - selected_reg_weight,
        selected_ic=round(selected_metrics["ic"], 6),
        selected_ir=round(selected_metrics["ir"], 6),
    )

    del reg_validation, rank_validation, fit_part, validation_part
    del reg_prediction, rank_prediction, reg_rank, pair_rank
    del fit_weights, validation_weights
    gc.collect()
    final_train = train_df.sort_values(
        ["date", "instrument"], kind="mergesort"
    ).reset_index(drop=True)
    final_groups = group_sizes(final_train)
    final_weights = (
        1.0 + 1.0 * np.abs(final_train["label"].to_numpy())
    ).astype("float32")

    reg_model = xgb.XGBRegressor(
        **model_parameters(reg_trees, "reg:squarederror"), eval_metric="rmse"
    )
    fit_t0 = time.time()
    _log(
        "reg_final_fit_start", rows=len(final_train), dates=len(final_groups),
        features=len(model_feature_cols), trees=reg_trees, df_mb=_df_mb(final_train),
    )
    reg_model.fit(
        final_train[model_feature_cols], final_train["label"],
        sample_weight=final_weights, verbose=False,
    )
    reg_importance = reg_model.get_booster().get_score(importance_type="gain")
    reg_top = sorted(reg_importance.items(), key=lambda item: item[1], reverse=True)[:10]
    _log(
        "reg_final_fit_done", seconds=round(time.time() - fit_t0, 2),
        top_features=";".join(f"{name}:{value:.3f}" for name, value in reg_top),
    )

    rank_model = xgb.XGBRanker(
        **model_parameters(rank_trees, "rank:pairwise"), eval_metric="ndcg@100"
    )
    fit_t0 = time.time()
    _log(
        "rank_final_fit_start", rows=len(final_train), groups=len(final_groups),
        features=len(model_feature_cols), trees=rank_trees,
    )
    rank_model.fit(
        final_train[model_feature_cols], final_train["rank_label"],
        group=final_groups, verbose=False,
    )
    rank_importance = rank_model.get_booster().get_score(importance_type="gain")
    rank_top = sorted(rank_importance.items(), key=lambda item: item[1], reverse=True)[:10]
    _log(
        "rank_final_fit_done", seconds=round(time.time() - fit_t0, 2),
        top_features=";".join(f"{name}:{value:.3f}" for name, value in rank_top),
    )
    del train_df, final_train, final_groups, final_weights
    gc.collect()
    _log("train_objects_released")

    _log("test_dataset_start")
    test_df, test_feature_cols = prepare_dataset(
        datasources["bar1m"],
        test_query_start.strftime("%Y-%m-%d 00:00:00"),
        test_end.strftime("%Y-%m-%d 23:59:59"),
        test_start, test_end, "test",
    )
    if test_feature_cols != model_feature_cols:
        raise ValueError("训练和测试特征列不一致")
    _log("predict_start", rows=len(test_df), features=len(model_feature_cols))
    reg_test = reg_model.predict(test_df[model_feature_cols])
    rank_test = rank_model.predict(test_df[model_feature_cols])
    reg_test_rank = cross_sectional_rank(test_df["date"], reg_test)
    pair_test_rank = cross_sectional_rank(test_df["date"], rank_test)
    test_df["factor"] = (
        selected_reg_weight * reg_test_rank
        + (1.0 - selected_reg_weight) * pair_test_rank
    ).astype("float32")
    factor_data = test_df[["date", "instrument", "factor"]].copy()
    factor_data["factor"] = pd.to_numeric(factor_data["factor"], errors="coerce")
    factor_data["factor"] = factor_data["factor"].replace([np.inf, -np.inf], np.nan)
    _log(
        "predict_done", rows=len(factor_data), missing=int(factor_data["factor"].isna().sum()),
        factor_mean=round(float(factor_data["factor"].mean()), 6),
        factor_std=round(float(factor_data["factor"].std()), 6), df_mb=_df_mb(factor_data),
    )

    _log("output_pool_query_start")
    output_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]}, compression=True,
    ).df()
    output_pool["date"] = pd.to_datetime(output_pool["date"]).dt.normalize()
    output_pool["instrument"] = output_pool["instrument"].astype(str)
    result = pd.merge(output_pool, factor_data, how="left", on=["date", "instrument"])
    result = result[["date", "instrument", "factor"]].copy()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    missing_before = int(result["factor"].isna().sum())
    result["factor"] = result.groupby("date", sort=False)["factor"].transform(
        lambda x: x.fillna(x.median())
    )
    result["factor"] = result["factor"].fillna(0.0).astype("float32")
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    result = result.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)
    _log(
        "main_done", rows=len(result), missing_before_fill=missing_before,
        missing_after_fill=int(result["factor"].isna().sum()),
        selected_reg_weight=selected_reg_weight,
        total_seconds=round(time.time() - main_t0, 2), df_mb=_df_mb(result),
    )
    return result


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    logger.info("因子计算完成", rows=len(factor_data))
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-08-01 05:43:30] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[AI051] main_enter | elapsed=0.00s | rss_peak_mb=216.23 | start_date=2024-01-01 00:00:00 | end_date=2024-12-31 23:59:59
[AI051] xgboost_import_start | elapsed=0.00s | rss_peak_mb=216.23
[AI051] imports_done | elapsed=0.48s | rss_peak_mb=289.22 | xgboost_version=1.7.3
[AI051] windows_resolved | elapsed=0.48s | rss_peak_mb=289.22 | train_start=2021-01-01 00:00:00 | train_end=2023-12-31 00:00:00 | train_query_start=2020-11-27 00:00:00 | test_start=2024-01-01 00:00:00 | test_end=2024-12-31 23:59:59
[AI051] train_dataset_start | elapsed=0.48s | rss_peak_mb=289.22
[AI051] pool_preload_start | elapsed=0.48s | rss_peak_mb=289.22 | phase=train | start=2020-11-27 00:00:00 | end=2023-12-31 23:59:59
[AI051] pool_preload_done | elapsed=0.80s | rss_peak_mb=506.23 | phase=train | rows=752000 | df_mb=53.07
[AI051] feature_chunk_start | elapsed=0.80s | rss_peak_mb=506.23 | phase=train | chunk=1 | start=2020-11-27 00:00: